# 004 R0 — Colab T4 / FPNet 真实前向诊断

手动逐格执行：确认设备 → 下载固定版本公开权重并读取 metadata → 上传本地参考定义与真实谱样本 → 前向诊断 → 保存 JSON。**不训练、不计算 MRR、不报告提分。** 前向成功只说明该环境下模型与输入可以运行。

1. 在 Colab 选择 GPU runtime（计划使用 T4，实际型号会记录）。本 notebook 不申请或切换设备。
2. 通过实际可用的 MCP 文件写入/传输工具（若提供），或由用户一次性手动上传，把以下两份本地文件放到 `/content/`；不继续使用 Computer Use 自动化上传：
   - `artifacts/004-research/reference-fpnet-v32.py`：原作者定义的本地摘录，保留出处与许可待核实说明；模型代码未嵌入本 notebook。
   - `artifacts/004-research/np_forward_sample.json`：16 张 `enveda-np-examples` 真实谱，16 个不同 shipped 结构，正负模式各 8；只是前向诊断样本，不是评估集。
3. 数据和上传文件留在私有 notebook/runtime；原始谱未写入 `.ipynb`。执行上传的 Python 文件前应人工确认是已检查的参考摘录。

参考：[作者 notebook v32](https://www.kaggle.com/code/prvsiyan/analog-propagation-casmi-2026-baseline/version/32)、[公开 checkpoint 数据集 v1](https://www.kaggle.com/datasets/prvsiyan/casmi26-fp-models-v2/versions/1)。权重的 CC0 标注不自动覆盖代码与原始训练数据许可。checkpoint 的真实训练暴露未知。


In [ ]:
# 仅安装缺少的运行依赖，不升级 Colab 已有的 torch/CUDA。
import importlib.util
import subprocess
import sys
missing = [name for name in ("torch", "numpy", "kagglehub")
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
print({"installed_missing": missing})


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc, hashlib, importlib.metadata, json, platform, statistics, time
import numpy as np
import torch
import kagglehub

SAMPLE_PATH = Path("/content/np_forward_sample.json")
REFERENCE_PATH = Path("/content/reference-fpnet-v32.py")
OUTPUT_DIR = Path("/content/004-r0-output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_HANDLE = "prvsiyan/casmi26-fp-models-v2/versions/1"
CHECKPOINT_NAMES = ("fp_single_s2.pt", "fp_merged_m1.pt")
# 默认只核对 single 模型的真实单谱输入；merged 的训练输入分布另做研究。
FORWARD_CHECKPOINT_NAMES = ("fp_single_s2.pt",)
REPEAT_FORWARDS = 3
SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

report = {
    "experiment": "004-R0 FPNet genuine-spectrum forward feasibility",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "scope": "Inference compatibility only; no training, accuracy, MRR or leaderboard claim.",
    "seed": SEED,
    "environment": {
        "python": sys.version, "platform": platform.platform(),
        "torch": torch.__version__, "numpy": np.__version__,
        "kagglehub": importlib.metadata.version("kagglehub"),
        "torch_cuda_build": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_count": torch.cuda.device_count(),
        "gpus": [],
    },
    "checkpoints": [], "forward_runs": [],
    "research_effect": "not evaluated",
}

def save_report():
    report["updated_utc"] = datetime.now(timezone.utc).isoformat()
    path = OUTPUT_DIR / "r0_feasibility.json"
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n")
    return path

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    report["environment"]["gpus"].append({
        "index": i, "name": p.name, "total_memory_bytes": p.total_memory,
        "compute_capability": [p.major, p.minor],
    })
save_report()
print(json.dumps(report["environment"], indent=2, ensure_ascii=False))
if not torch.cuda.is_available():
    raise RuntimeError("当前 runtime 没有 CUDA GPU；环境已记录。请人工选择 GPU 后重跑，不能据此报告 T4 前向成功。")
DEVICE = torch.device("cuda:0")


下一格会下载公开数据集固定版本的两份权重（约 288 MB），只在 CPU 上读取其 metadata。加载始终使用 `weights_only=True`，不会自动回退到不受限 pickle 加载。加载失败会写入 JSON，并保留为未解决的兼容性结果。


In [ ]:
report["dataset"] = {"handle": DATASET_HANDLE}
t0 = time.perf_counter()
try:
    weights_dir = Path(kagglehub.dataset_download(DATASET_HANDLE))
except Exception as exc:
    report["dataset"]["download_error"] = {"type": type(exc).__name__, "message": str(exc)[:2000]}
    save_report()
    raise
report["dataset"]["download_seconds"] = time.perf_counter() - t0
report["dataset"]["resolved_path"] = str(weights_dir)
checkpoints = {}
for name in CHECKPOINT_NAMES:
    entry = {"filename": name, "weights_only": True}
    report["checkpoints"].append(entry)
    try:
        candidates = list(weights_dir.rglob(name))
        if len(candidates) != 1:
            raise FileNotFoundError(f"Expected one {name}, found {len(candidates)}")
        path = candidates[0]
        entry.update({"bytes": path.stat().st_size, "sha256": sha256_file(path)})
        t0 = time.perf_counter()
        ck = torch.load(path, map_location="cpu", weights_only=True)
        entry["load_cpu_seconds"] = time.perf_counter() - t0
        entry["container_type"] = type(ck).__name__
        entry["keys"] = list(ck.keys()) if isinstance(ck, dict) else None
        if not isinstance(ck, dict) or not isinstance(ck.get("model"), dict):
            raise ValueError("Expected checkpoint mapping with model state_dict")
        entry["metadata"] = {k: int(ck[k]) for k in ("step", "nbits", "d", "layers") if k in ck}
        entry["state_tensor_count"] = sum(torch.is_tensor(v) for v in ck["model"].values())
        entry["state_tensor_bytes"] = sum(v.numel() * v.element_size() for v in ck["model"].values() if torch.is_tensor(v))
        entry["status"] = "loaded_on_cpu"
        checkpoints[name] = ck
    except Exception as exc:
        entry["status"] = "load_failed"
        entry["error"] = {"type": type(exc).__name__, "message": str(exc)[:2000]}
    save_report()
    print(json.dumps(entry, indent=2, ensure_ascii=False))


In [ ]:
# 读取人工上传的文件；不从 notebook 内嵌谱数据或复制第三方模型定义。
missing_inputs = [str(p) for p in (REFERENCE_PATH, SAMPLE_PATH) if not p.is_file()]
if missing_inputs:
    report["input_status"] = {"missing_files": missing_inputs}
    save_report()
    raise FileNotFoundError("请通过 Colab 文件侧栏上传：" + ", ".join(missing_inputs))
report["reference"] = {
    "path": str(REFERENCE_PATH), "sha256": sha256_file(REFERENCE_PATH),
    "source": "https://www.kaggle.com/code/prvsiyan/analog-propagation-casmi-2026-baseline/version/32",
    "license_status": "Check original notebook code license; checkpoint license is separate.",
}
reference_namespace = {"__name__": "fpnet_reference", "__file__": str(REFERENCE_PATH)}
try:
    exec(compile(REFERENCE_PATH.read_text(), str(REFERENCE_PATH), "exec"), reference_namespace)
    FPNet = reference_namespace["FPNet"]
    prep_peaks = reference_namespace["prep_peaks"]
    ADDUCT_IX = reference_namespace["ADDUCT_IX"]
    instr_family = reference_namespace["instr_family"]
    MAX_PEAKS = int(reference_namespace["MAX_PEAKS"])
except Exception as exc:
    report["reference"]["execution_error"] = {"type": type(exc).__name__, "message": str(exc)[:2000]}
    save_report()
    raise
sample = json.loads(SAMPLE_PATH.read_text())
records = sample["spectra"]
report["sample"] = {
    "path": str(SAMPLE_PATH), "sha256": sha256_file(SAMPLE_PATH),
    "source": sample.get("source"), "selection": sample.get("selection"),
    "actual_records": len(records), "use": "forward diagnostic only",
}
save_report()
print({"reference_sha256": report["reference"]["sha256"], "sample_records": len(records)})


In [ ]:
# 使用参考文件中的峰预处理与类别映射；本格只适配真实 JSON 记录为固定 128 峰张量。
# 原始 m/z 在 prep_peaks 内以 float64 处理，模型张量使用 float32；本轮不启用 AMP/BF16。
prepared, skipped, ce_fallback_ids = [], [], []
for row in records:
    sid = row["sample_id"]
    try:
        raw_mz = np.asarray(row["ms2_mzs"], dtype=np.float64)
        raw_it = np.asarray(row["ms2_normalized_intensities"], dtype=np.float64)
        precursor = float(row["precursor_mz"])
        if raw_mz.ndim != 1 or raw_it.ndim != 1 or len(raw_mz) != len(raw_it):
            raise ValueError("Unpaired or non-vector peak arrays")
        if not np.isfinite(raw_mz).all() or not np.isfinite(raw_it).all() or not np.isfinite(precursor):
            raise ValueError("Nonfinite raw peak or precursor")
        if (raw_it < 0).any():
            raise ValueError("Negative peak intensity")
        if row["ionization_mode"] not in ("positive", "negative"):
            raise ValueError("Unknown ionization_mode")
        mz, intensity = prep_peaks(raw_mz, raw_it, precursor, max_peaks=MAX_PEAKS)
        if not len(mz):
            raise ValueError("Reference preprocessing removed all peaks")
        if not np.isfinite(mz).all() or not np.isfinite(intensity).all():
            raise ValueError("Nonfinite preprocessed peak")
        ce_value = row.get("collision_energy_ev")
        try:
            ce_array = np.asarray(ce_value, dtype=np.float64).reshape(-1)
            if ce_value is None or not len(ce_array) or not np.isfinite(ce_array).all():
                raise ValueError("Missing collision energy")
            ce = float(ce_array.mean())
        except (TypeError, ValueError):
            ce = 25.0
            ce_fallback_ids.append(sid)
        prepared.append((row, mz, intensity, precursor, ce))
    except Exception as exc:
        skipped.append({"sample_id": sid, "reason": str(exc)})
B = len(prepared)
report["preprocessing"] = {
    "max_peaks": MAX_PEAKS, "valid_records": B, "skipped": skipped,
    "ce_fallback_ev": 25.0, "ce_fallback_sample_ids": ce_fallback_ids,
    "mode_mapping": {"positive": 1.0, "negative": -1.0},
    "adduct_mapping": "Reference ADDUCT_IX; unknown -> <unk>",
    "source_sample_ids": [r[0]["sample_id"] for r in prepared],
    "peak_counts_after_prep": [len(r[1]) for r in prepared],
    "input_view": "one real spectrum per distinct structure; no spectrum merging",
}
save_report()
if not B:
    raise ValueError("没有可用真实谱，已保存具体原因；不使用随机或零样本替代。")
mz_np = np.zeros((B, MAX_PEAKS), dtype=np.float32)
it_np = np.zeros_like(mz_np)
pad_np = np.ones((B, MAX_PEAKS), dtype=bool)
for i, (_, mz, intensity, _, _) in enumerate(prepared):
    mz_np[i, :len(mz)] = mz
    it_np[i, :len(mz)] = intensity
    pad_np[i, :len(mz)] = False
cpu_args = (
    torch.from_numpy(mz_np), torch.from_numpy(it_np), torch.from_numpy(pad_np),
    torch.tensor([r[3] for r in prepared], dtype=torch.float32),
    torch.tensor([ADDUCT_IX.get(r[0]["adduct"], ADDUCT_IX["<unk>"]) for r in prepared], dtype=torch.long),
    torch.tensor([instr_family(r[0]["instrument_type"]) for r in prepared], dtype=torch.long),
    torch.tensor([r[4] for r in prepared], dtype=torch.float32),
    torch.tensor([1.0 if r[0]["ionization_mode"] == "positive" else -1.0 for r in prepared], dtype=torch.float32),
)
report["preprocessing"]["tensor_shapes"] = [list(x.shape) for x in cpu_args]
report["preprocessing"]["tensor_dtypes"] = [str(x.dtype) for x in cpu_args]
save_report()
print(json.dumps(report["preprocessing"], indent=2, ensure_ascii=False))


下面是人工前向入口：默认仅运行 `fp_single_s2.pt`，输入为上述真实单谱。`fp_merged_m1.pt` 已做安全加载与 metadata 检查，但这不等于验证了它的合并谱输入路径。若人工把 merged 文件加入列表，本 notebook 会明确记录它接收的是单谱，仅可用于 shape/算子兼容性观察，不能比较模型质量或据此选融合方法。

执行 `eval()`、`inference_mode()`；首个前向与后续 3 次前向分别计时。这个小 batch 的耗时只反映当前诊断，不估计完整训练吞吐。没有 optimizer、backward 或训练循环。


In [ ]:
# 手动运行本格，执行少量真实谱前向，不开展训练。
for name in FORWARD_CHECKPOINT_NAMES:
    run = {"filename": name, "device": str(DEVICE), "dtype": "float32", "mode": "eval/inference_mode",
           "input_view": "single real spectra", "trained_input_view_match": "merged" not in name,
           "batch_size": B, "repeat_forward_count": REPEAT_FORWARDS}
    report["forward_runs"].append(run)
    model, device_args, logits = None, None, None
    try:
        if name not in checkpoints:
            raise RuntimeError("Checkpoint did not load safely; see checkpoints errors")
        ck = checkpoints[name]
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize(DEVICE)
        run["memory_before_model_bytes"] = torch.cuda.memory_allocated(DEVICE)
        torch.cuda.reset_peak_memory_stats(DEVICE)
        t0 = time.perf_counter()
        # heads=8/drop=0.1 are reference defaults; dimensions come from checkpoint metadata.
        model = FPNet(nbits=int(ck["nbits"]), d=int(ck["d"]), layers=int(ck["layers"]), heads=8, drop=0.1)
        incompatible = model.load_state_dict(ck["model"], strict=True)
        run["state_dict"] = {"strict": True, "missing_keys": list(incompatible.missing_keys),
                             "unexpected_keys": list(incompatible.unexpected_keys)}
        model = model.to(DEVICE).eval()
        device_args = tuple(x.to(DEVICE) for x in cpu_args)
        torch.cuda.synchronize(DEVICE)
        run["construct_load_transfer_seconds"] = time.perf_counter() - t0
        run["parameter_count"] = sum(p.numel() for p in model.parameters())
        with torch.inference_mode():
            t0 = time.perf_counter()
            logits = model(*device_args)
            torch.cuda.synchronize(DEVICE)
            run["first_forward_seconds"] = time.perf_counter() - t0
            times = []
            for _ in range(REPEAT_FORWARDS):
                t0 = time.perf_counter()
                logits = model(*device_args)
                torch.cuda.synchronize(DEVICE)
                times.append(time.perf_counter() - t0)
        values = logits.detach().float().cpu().numpy()
        finite = np.isfinite(values)
        finite_values = values[finite]
        run.update({
            "status": "forward_completed", "output_shape": list(values.shape),
            "expected_shape": [B, int(ck["nbits"])],
            "output_shape_matches": list(values.shape) == [B, int(ck["nbits"])],
            "all_outputs_finite": bool(finite.all()),
            "finite_values": int(finite.sum()), "nonfinite_values": int((~finite).sum()),
            "output_min": float(finite_values.min()) if len(finite_values) else None,
            "output_max": float(finite_values.max()) if len(finite_values) else None,
            "output_mean": float(finite_values.mean()) if len(finite_values) else None,
            "repeat_forward_seconds": times,
            "median_repeat_seconds": statistics.median(times) if times else None,
            "peak_cuda_allocated_bytes": torch.cuda.max_memory_allocated(DEVICE),
            "peak_cuda_reserved_bytes": torch.cuda.max_memory_reserved(DEVICE),
            "memory_note": "Peak since reset before model construction/transfer; includes model, input and inference temporaries.",
        })
    except Exception as exc:
        run["status"] = "forward_failed"
        run["error"] = {"type": type(exc).__name__, "message": str(exc)[:4000]}
        if torch.cuda.is_available():
            run["peak_cuda_allocated_bytes"] = torch.cuda.max_memory_allocated(DEVICE)
    finally:
        save_report()
        print(json.dumps(run, indent=2, ensure_ascii=False))
        del model, device_args, logits
        gc.collect()
        torch.cuda.empty_cache()
print("Diagnostic complete. No training or research-effect conclusion was produced.")


In [ ]:
# 结果文件不包含原始谱、模型权重或预测准确率。通过 Colab 文件侧栏下载本 JSON。
report["completed_utc"] = datetime.now(timezone.utc).isoformat()
report_path = save_report()
print("Saved:", report_path)
print({"weights_loaded_on_cpu": list(checkpoints),
       "forward_states": [{"filename": x["filename"], "status": x.get("status"),
                           "shape": x.get("output_shape"), "finite": x.get("all_outputs_finite")}
                          for x in report["forward_runs"]],
       "training": "not run", "research_effect": "not evaluated"})


将 `/content/004-r0-output/r0_feasibility.json` 下载到本地 `artifacts/004-research/`，再把设备、依赖、权重/输入 hash、前向形状、有限值、计时、峰值显存与失败原因记录到本轮 progress。不要把 `forward_completed` 直接解释为输出有效：还要看 shape、finite 和错误字段。

下一步研究依赖官方指纹位映射、正确的候选池与独立留出设计；此 notebook 不生成 submission，不使用 16 谱样本评估泛化，也没有 TPU 训练实现。
